In [3]:
import os
print(os.getcwd())
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import DataLoader
from torch.optim import Adam,AdamW
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from utils import simdatset, reproducibility

/disk1/user/liaoshuilin/project/35.TAPE_EXO/assay_diffusion


# Load data

In [4]:
import pickle

with open(f'../result/data_sigComp/Stim_data.pkl', 'rb') as file:
    loaded_data = pickle.load(file)

GTE_x_train = loaded_data['GTE_x_train']
GTE_x_val = loaded_data['GTE_x_val']
GTE_x_test = loaded_data['GTE_x_test']
GTE_y_train = loaded_data['GTE_y_train']
GTE_y_val = loaded_data['GTE_y_val']
GTE_y_test = loaded_data['GTE_y_test']

HPA_x = loaded_data['HPA_x']
HPA_y = loaded_data['HPA_y']
real_x = loaded_data['real_x']

genes = loaded_data['genes']
celltypes = loaded_data['celltypes']

print(GTE_x_train.shape)

(4000, 712)


# Model DADA-EV

In [6]:
%load_ext autoreload
%autoreload 2
from train_re import AdaptiveTAPEandDiffusion2, alternate_training_earlyStop, evaluation, adaptive_stage_domain9
from utils import reproducibility, calculate_evaluation_metrics

batch_size = 256
reproducibility(2025)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
out_pth = "../result/model_sigComp/"

## stage1 model
model = AdaptiveTAPEandDiffusion2(GTE_x_train.shape[1], GTE_y_train.shape[1],2, T=2000).to(device)

optimizer_main = AdamW([
    {'params': model.encoder.parameters()},
    {'params': model.predictor.parameters()},
    {'params': model.decoder.parameters()}], lr=1e-4)
optimizer_diffusion =AdamW(model.ref_creator.parameters(), lr=1e-3)
optimizer_all = AdamW(model.parameters(),1e-4)
epochs_main = 400 
epochs_diffusion = 1500

train_loader = DataLoader(simdatset(GTE_x_train, GTE_y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(simdatset(GTE_x_val, GTE_y_val), batch_size=batch_size, shuffle=False)

model, main_loss, diffloss = alternate_training_earlyStop(model, train_loader, val_loader, optimizer_main, optimizer_diffusion, 
                                                 epochs_main, epochs_diffusion, device='cpu',
                                                 patience=5, early_stop_start=400, early_stop_interval=50)
torch.save(model, out_pth + "model_stage1.pth")
# model = torch.load(out_pth + "model_stage1.pth", weights_only=False, map_location=device)

## sigmatrix
train_loader2 = DataLoader(simdatset(GTE_x_train, GTE_y_train), batch_size=batch_size, shuffle=False)
x_recon_tr, f_tr, z_tr  = evaluation(train_loader2, model, device=device)
sigmatrix = np.linalg.pinv(f_tr) @ x_recon_tr 
pd.DataFrame(sigmatrix).to_csv(out_pth + 'sigmatrix.csv', index=True, header=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[Warmup] epochs = 50
[Warmup phas 1 Epoch 1/50] loss: 0.7433
[Warmup phas 1 Epoch 2/50] loss: 0.7107
[Warmup phas 1 Epoch 3/50] loss: 0.7660
[Warmup phas 1 Epoch 4/50] loss: 0.7146
[Warmup phas 1 Epoch 5/50] loss: 0.7010
[Warmup phas 1 Epoch 6/50] loss: 0.7348
[Warmup phas 1 Epoch 7/50] loss: 0.7129
[Warmup phas 1 Epoch 8/50] loss: 0.7341
[Warmup phas 1 Epoch 9/50] loss: 0.7130
[Warmup phas 1 Epoch 10/50] loss: 0.7255
[Warmup phas 1 Epoch 11/50] loss: 0.7233
[Warmup phas 1 Epoch 12/50] loss: 0.7215
[Warmup phas 1 Epoch 13/50] loss: 0.7179
[Warmup phas 1 Epoch 14/50] loss: 0.7044
[Warmup phas 1 Epoch 15/50] loss: 0.7244
[Warmup phas 1 Epoch 16/50] loss: 0.7005
[Warmup phas 1 Epoch 17/50] loss: 0.7264
[Warmup phas 1 Epoch 18/50] loss: 0.7079
[Warmup phas 1 Epoch 19/50] loss: 0.7247
[Warmup phas 1 Epoch 20/50] loss: 0.7134
[Warmup phas 1 Epoch 21/50] loss: 0.7378
[Warmup phas 1 Epoch 22/50] loss: 0.698

In [7]:
## AE
test_loader = DataLoader(simdatset(GTE_x_test, GTE_y_test), batch_size=batch_size, shuffle=False)
x_recon_te, f_te, z_te  = evaluation(test_loader, model, device=device)
pd.DataFrame(x_recon_te).to_csv(out_pth + 'GTE_AE_x_recon.csv', index=False, header=False)
pd.DataFrame(f_te).to_csv(out_pth + 'GTE_AE_y.csv', index=False, header=False)
rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = x_recon_te, input_X = GTE_x_test, out_pth = out_pth + "GTE_AE_x_")
print(f"AE\nRMSE of test x: {average_rmse:.4f}\nPCC of x: {average_pearson_corr:.4f}\nMAE of x: {average_mae:.4f}\nCCC of x: {average_ccc:.4f}")
rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = f_te, input_X = GTE_y_test, out_pth = out_pth + "GTE_AE_y_")
print(f"RMSE of test frac: {average_rmse:.4f}\nPCC of frac: {average_pearson_corr:.4f}\nMAE of frac: {average_mae:.4f}\nCCC of frac: {average_ccc:.4f}")

AE
RMSE of test x: 0.0316
PCC of x: 0.9879
MAE of x: 0.0242
CCC of x: 0.9836
RMSE of test frac: 0.0145
PCC of frac: 0.9638
MAE of frac: 0.0093
CCC of frac: 0.9516


In [ ]:
## DADA
x_recon_HPA, f_HPA, z_HPA, model2 = adaptive_stage_domain9(x=HPA_x, model_name=out_pth + "model_stage1", mode = 'overall5', steps=10, max_iter=40, device=device, sigmatrix = sigmatrix)
pd.DataFrame(x_recon_HPA).to_csv(out_pth + 'HPA_DADA_x_recon.csv', index=False, header=False)
pd.DataFrame(f_HPA).to_csv(out_pth + 'HPA_DADA_y.csv', index=False, header=False)
rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = x_recon_HPA, input_X = HPA_x, out_pth = out_pth + "HPA_DADA_x_")
print(f"DADA\nRMSE of x: {average_rmse:.4f}\nPCC of x: {average_pearson_corr:.4f}\nMAE of x: {average_mae:.4f}\nCCC of x: {average_ccc:.4f}")
rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = f_HPA, input_X = HPA_y, out_pth = out_pth + "HPA_DADA_y_")
print(f"RMSE of test frac: {average_rmse:.4f}\nPCC of frac: {average_pearson_corr:.4f}\nMAE of frac: {average_mae:.4f}\nCCC of frac: {average_ccc:.4f}")
torch.save(model2, out_pth + "model_stage2.pth")

In [9]:
## sigmatrix
mode2 = torch.load(out_pth + "model_stage2.pth", weights_only=False, map_location=device)
train_loader2 = DataLoader(simdatset(HPA_x, HPA_y), batch_size=batch_size, shuffle=False)
x_recon_HPA, f_HPA, z_HPA  = evaluation(train_loader2, mode2, device=device)
sigmatrix2 = np.linalg.pinv(f_HPA) @ x_recon_HPA 
pd.DataFrame(sigmatrix2).to_csv(out_pth + 'sigmatrix_encoder2.csv', index=True, header=True)

# write csv

In [8]:
## AE
x_recon_te = pd.read_csv(out_pth + 'GTE_AE_x_recon.csv', header=None)
f_te =  pd.read_csv(out_pth + 'GTE_AE_y.csv', header=None)

rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = x_recon_te, input_X = GTE_x_test, out_pth = out_pth + "GTE_AE_x_")
print(f"AE\nRMSE of test x: {average_rmse:.4f}\nPCC of x: {average_pearson_corr:.4f}\nMAE of x: {average_mae:.4f}\nCCC of x: {average_ccc:.4f}")
rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = f_te, input_X = GTE_y_test, out_pth = out_pth + "GTE_AE_y_")
print(f"RMSE of test frac: {average_rmse:.4f}\nPCC of frac: {average_pearson_corr:.4f}\nMAE of frac: {average_mae:.4f}\nCCC of frac: {average_ccc:.4f}")

## DADA
x_recon_HPA = pd.read_csv(out_pth + 'HPA_DADA_x_recon.csv', header=None)
f_HPA =  pd.read_csv(out_pth + 'HPA_DADA_y.csv', header=None)

rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = x_recon_HPA, input_X = HPA_x, out_pth = out_pth + "HPA_DADA_x_")
print(f"DADA\nRMSE of x: {average_rmse:.4f}\nPCC of x: {average_pearson_corr:.4f}\nMAE of x: {average_mae:.4f}\nCCC of x: {average_ccc:.4f}")
rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = f_HPA, input_X = HPA_y, out_pth = out_pth + "HPA_DADA_y_")
print(f"RMSE of test frac: {average_rmse:.4f}\nPCC of frac: {average_pearson_corr:.4f}\nMAE of frac: {average_mae:.4f}\nCCC of frac: {average_ccc:.4f}")

AE
RMSE of test x: 0.0316
PCC of x: 0.9879
MAE of x: 0.0242
CCC of x: 0.9836
RMSE of test frac: 0.0145
PCC of frac: 0.9638
MAE of frac: 0.0093
CCC of frac: 0.9516
DADA
RMSE of x: 0.0336
PCC of x: 0.9867
MAE of x: 0.0250
CCC of x: 0.9834
RMSE of test frac: 0.0250
PCC of frac: 0.8962
MAE of frac: 0.0157
CCC of frac: 0.8673
